# Malaria Cell Image Classification with CNN & Transfer Learning

Bu projede, NIH (National Institutes of Health) tarafından sağlanan Malaria veri setini kullanarak, hücre görüntülerinden hücrenin "Enfekte (Parasitized)" mi yoksa "Enfekte Olmamış (Uninfected)" mı olduğunu tespit eden derin öğrenme modelleri geliştireceğiz.

Proje iki ana aşamadan oluşacaktır:
- Custom CNN: Kendi tasarladığımız evrişimli sinir ağı modeli.
- Transfer Learning: Önceden eğitilmiş (VGG16, ResNet50) modellerin bu veriye uyarlanması.

In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import itertools
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

2025-11-29 07:09:00.032917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764400140.271418      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764400140.339952      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version: 2.18.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
# Ayarlar
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 32
IMAGE_SIZE = (128, 128)   # Transfer learning için istersen 224x224 yapabiliriz
EPOCHS = 20
AUTOTUNE = tf.data.AUTOTUNE

In [8]:
BASE = '/kaggle/input/cell-images-for-detecting-malaria/cell_images'

# İç içe klasör varsa
if os.path.exists(os.path.join(BASE, "cell_images")):
    ROOT_DIR = os.path.join(BASE, "cell_images")
else:
    ROOT_DIR = BASE

print("Loading from:", ROOT_DIR)
print(os.listdir(ROOT_DIR))

Loading from: /kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images
['Uninfected', 'Parasitized']


In [9]:
# image_dataset_from_directory ile veri yükleme (train/val split)
# Bu yöntem batch olarak görüntüleri disktan okur; dolayısıyla belleğe hepsini almıyoruz.
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    ROOT_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    ROOT_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Found 27558 files belonging to 2 classes.
Using 22047 files for training.
Found 27558 files belonging to 2 classes.
Using 5511 files for validation.
Classes: ['Parasitized', 'Uninfected']


In [10]:
# CNN modeli
def make_baseline_model(input_shape=(*IMAGE_SIZE,3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(class_names), activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='baseline_cnn')
    return model

model = make_baseline_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,287,938 (16.36 MB)

 Trainable params: 4,287,938 (16.36 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Model egitimi
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history_baseline = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.7009 - loss: 0.5370 - val_accuracy: 0.9401 - val_loss: 0.1748 - learning_rate: 0.0010
Epoch 2/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9506 - loss: 0.1543 - val_accuracy: 0.9577 - val_loss: 0.1311 - learning_rate: 0.0010
Epoch 3/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9582 - loss: 0.1336 - val_accuracy: 0.9526 - val_loss: 0.1358 - learning_rate: 0.0010
Epoch 4/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.9598 - loss: 0.1197 - val_accuracy: 0.9505 - val_loss: 0.1450 - learning_rate: 0.0010
Epoch 5/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.9626 - loss: 0.1098 - val_accuracy: 0.9565 - val_loss: 0.1307 - learning_rate: 0.0010
Epoch 6/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.9647 - loss: 0.1000 - val_accuracy: 0.9517 - val_loss: 0.1355 - learning_rate: 0.0010
Epoch 7/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9694 - l

### Sonuc:
Egitim yapilirken earlyStopping yapilarak 0.98 accuracy skor ve 0.95 val_accuracy skor elde edildi. Bizim icin yeterli, simdi modeli kaydedip transfer learning yapalim.

In [12]:
model.save('baseline_cnn_malaria.h5')
model.save('baseline_cnn_malaria.keras')

### Transfer Learning

In [16]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

In [17]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [18]:
s_model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

In [19]:
s_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [20]:
history_vgg16 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 18s 26ms/step - accuracy: 0.9670 - loss: 0.0978 - val_accuracy: 0.9554 - val_loss: 0.1284
Epoch 2/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.9705 - loss: 0.0813 - val_accuracy: 0.9590 - val_loss: 0.1401
Epoch 3/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.9755 - loss: 0.0699 - val_accuracy: 0.9559 - val_loss: 0.1433
Epoch 4/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.9787 - loss: 0.0600 - val_accuracy: 0.9552 - val_loss: 0.1558
Epoch 5/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.9802 - loss: 0.0532 - val_accuracy: 0.9559 - val_loss: 0.1824
Epoch 6/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.9870 - loss: 0.0406 - val_accuracy: 0.9559 - val_loss: 0.1971
Epoch 7/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.9894 - loss: 0.0289 - val_accuracy: 0.9557 - val_loss: 0.2430
Epoch 8/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.9921 - loss: 0.0216 - 

### Sonuc
Buradada VGG16 modelini transfer learning yaparak 0.99 accuracy skor ve 0.95 val_accuracy skor elde ettik, Resnet modelini test etmeye gerek kalmadi. Bu modeli kaydedip kullanabiliriz.

In [21]:
model.save('VGG16_cnn_malaria.h5')
model.save('VGG16_cnn_malaria.keras')